# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [84]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# List to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None  # Default to None (indicating missing content)
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)  # Log failed download

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 467
n_rows = 100+1 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"], 
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")


/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Fetching .mod files:   4%|▍         | 4/101 [00:00<00:06, 15.69it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod


Fetching .mod files:  33%|███▎      | 33/101 [00:02<00:03, 17.46it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=38229&file=cahva_chan.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38229&file=cahva_chan.mod
🚨 Skipping invalid URL: https://modeldb.science/2016666-old?tab=2&file=popovic2015/locator.mod


Fetching .mod files:  62%|██████▏   | 63/101 [00:04<00:02, 14.37it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/Mykca.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/Mykca.mod


Fetching .mod files: 100%|██████████| 101/101 [00:07<00:00, 14.35it/s]

⚠️ Some downloads failed. Check ../data/failed_downloads.json for details.

✅ All .mod files (including failures) saved in ../data/mod_files.json


# Read JSON file

In [85]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
pd.set_option("display.max_columns", None)

In [86]:
def View(df, rows=5, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [95]:
# Function to extract mod directory from the URL
def extract_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def extract_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to determine exclusion reason with snake_case labels
def determine_exclusion(row):
    if pd.isna(row["content"]):
        return "not_found"  # Standardized exclusion for missing files
    if "x86_64" in row["url"]:
        return "x86_64"  # Standardized exclusion for architecture issues
    return None  # Keep valid entries as None (not excluded)

# Function to extract all TITLE occurrences from .mod content
def extract_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None


# Function to extract all COMMENT sections from .mod content
def extract_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def extract_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all USEION occurrences, stopping before VALENCE
def extract_use_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"USEION\s+([^\s]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    return list(set(matches)) if matches else None  # Remove duplicates


# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, GLOBAL, NONSPECIFIC_CURRENT, or VALENCE
def extract_read_ion(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  


# Function to extract all ions listed after WRITE, stopping before VALENCE
def extract_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  


def write_current_yn(ions):
    """
    Checks if mod_write_ion contains an ion that starts with 'i' (indicating a current).

    Args:
        write_ions (list): List of ions written in the mod file.

    Returns:
        int: 1 if any ion starts with 'i', otherwise 0.
    """
    if not ions:  # Handle empty lists or None
        return 0

    return int(any(ion.startswith("i") for ion in ions))


# Function to extract all NONSPECIFIC currents
def extract_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def extract_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def extract_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def extract_parameter(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"PARAMETER\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    param_dict = {}
    
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            param_match = re.match(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
            if param_match:
                param_name, param_value = param_match.groups()
                param_dict[param_name] = float(param_value)  

    return param_dict if param_dict else None  

# Function to extract only active STATE variables, ignoring comments (`:`) and unit values `(mV)`, etc.
def extract_state(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore fully commented-out lines
                continue
            line = re.split(r"\s*:\s*", line)[0]  # Remove inline comments (anything after `:`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()  # Remove unit values
            if clean_line:
                state_vars.append(clean_line)

    return state_vars if state_vars else None  


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def extract_global(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  

    
# Function to extract webpage heading
def extract_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def extract_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def extract_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def extract_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found

In [96]:
# Load JSON into DataFrame
df = pd.read_json("../data/mod_files.json")

# Set "row_id" as the index
df.set_index("row_id", inplace=True)

# Exclude mod files
df["mod_exclude"] = df.apply(determine_exclusion, axis=1)  # Apply exclusion function

# Extract features from url
df["mod_heading"] = df["url"].apply(extract_heading)  # Extract heading from webpage
df["mod_citation"] = df["mod_heading"].apply(extract_citation)
df["mod_first_author"] = df["mod_citation"].apply(extract_author)
df["mod_year"] = df["mod_citation"].apply(extract_year)  # Now handles multiple years


df["mod_dir"] = df["url"].apply(extract_dir)
df["mod_name"] = df["url"].apply(extract_fname)
df["mod_model_id"] = df["url"].apply(extract_model_id)

#  Extract features from content
df["mod_title"] = df["content"].apply(extract_title)
df["mod_comment"] = df["content"].apply(extract_comment)
df["mod_suffix"] = df["content"].apply(extract_suffix)
df["mod_use_ion"] = df["content"].apply(extract_use_ion)
df["mod_read_ion"] = df["content"].apply(extract_read_ion)
df["mod_write_ion"] = df["content"].apply(extract_write_ion)
df["write_current"] = df["mod_write_ion"].apply(write_current_yn)
df["mod_nonspecific_current"] = df["content"].apply(extract_nonspecific_current)
df["mod_range"] = df["content"].apply(extract_range)
df["mod_global"] = df["content"].apply(extract_global)
df["mod_parameter"] = df["content"].apply(extract_parameter)
df["mod_state"] = df["content"].apply(extract_state)


In [99]:
df.loc[468:478, ["mod_read_ion"]]

,mod_read_ion
row_id,
468,None
469,None
470,[ena]
471,None
472,[ek]
473,None
474,[ek]
475,None
476,None


In [90]:
View(df.loc[470,["url"]])

url    https://modeldb.science/136026?tab=2&file=djurisic2008/na.mod
Name: 470, dtype: object

In [47]:
#df[(df["mod_exclude"] == "not_found") | (df["mod_exclude"] == "x86_64")].shape

# Flagging Issues

In [6]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)
#row_id: 483 - was only extracting ONE parameter instead of multiple parameters (fixed)
#row_id 31 - has VALENCE in the write_ion (https://modeldb.science/116862?tab=2&file=b09jan13/IL3.mod)

# Questions

In [ ]:
#Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? 
#what's the best way to capture 
"""example of a function that captures both active and commented out variables. I think its cleaner to capture active only?
import re
import pandas as pd

# Function to extract RANGE variables based on mode
def extract_range(content, mode="all"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")

"""

In [9]:
View(df.loc[481,["url","mod_state"]],50)

url                                                                                                                                                                                                                                                                                                                                                                                 https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod
mod_state    [A_AMPA       : AMPA state variable to construct the dual-exponential profile - decays with conductance tau_r_AMPA, B_AMPA       : AMPA state variable to construct the dual-exponential profile - decays with conductance tau_d_AMPA, A_NMDA       : NMDA state variable to construct the dual-exponential profile - decays with conductance tau_r_NMDA, B_NMDA       : NMDA state variable to construct the dual-exponential profile - decays with conductance tau_d_NMDA]
Name: 481, dtype: object

In [65]:
import re





In [64]:
! git add .
! git commit -m "convert"
! git push 

[main 13db747] modified read, write, use ion to ignore VALENCE
 1 file changed, 1252 insertions(+), 917 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 36 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 13.99 KiB | 4.66 MiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To github.com:innacohen/mod-extract.git
   8c4a8a4..13db747  main -> main


In [ ]:
import re




In [46]:
extract_nonspecific_comments(sample)

In [66]:
sample = """ 
NEURON {
	SUFFIX ical3
	USEION Ca READ Cai, Cao WRITE iCa VALENCE 2
        RANGE pcabar, g
	GLOBAL 	m_inf, tau_m, sh1, sh2, rat
}

"""

In [71]:
extract_read_ion(sample)

{'concentration': ['Cai', 'Cao']}